# Notebook de Test - Pipeline Consommation Électrique

Test pas-à-pas des tâches du pipeline de consommation.
Utilise les configurations par environnement (ENV=dev|test|prod).

## 1. Initialisation et Configuration

In [8]:
import os
import logging
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# Charger les variables d'environnement
load_dotenv(find_dotenv(".env"), override=True)
load_dotenv(find_dotenv(".env.secrets"), override=True)

# Configurer le logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
from ml.config import load_config

# Definir l'environnement de test
os.environ['ENV'] = 'test'

# Définir l'environnement (dev par défaut)
env = os.getenv('ENV', 'test') # 'dev', 'test' ou 'prod'
print(f'Environment: {env}')

# Charger la configuration
#CONFIG_PATH = "../src/configs/consumption.yaml"
config = load_config(config_name="consumption")

# Afficher la configuration chargée
print('=== CONFIGURATION ===')
print(f"Raw file: {config.get('data', {}).get('raw_path')}")
print(f"Processed path: {config.get('data', {}).get('processed_path')}")
print(f"Target column: {config.get('data', {}).get('target_column')}")
print(f"Feature columns: {config.get('data', {}).get('feature_columns')}")
print(f"Model type: {config.get('model', {}).get('model_type')}")

Environment: test
=== CONFIGURATION ===
Raw file: ../data/test/raw_consumption.csv
Processed path: ../data/test/
Target column: Valeur
Feature columns: ['temperature_2m_mean', 'relative_humidity_mean', 'precipitation_sum', 'is_vacances', 'jour de la semaine', 'jour férié']
Model type: auto_gluon


## 2. Test du ConsumptionDataPreparer

In [10]:
from ml.consumption.consumption_preparer import ConsumptionDataPreparer

# Initialiser le preparer (utilise la config par défaut)
preparer = ConsumptionDataPreparer()
print('✅ ConsumptionDataPreparer initialisé')

✅ ConsumptionDataPreparer initialisé


### 2. Chargement des données consommation (avec raw_path depuis config)

In [11]:
# Tester le chargement avec raw_path depuis la config
# Ou intégrér test_file = '../data/test/test_raw_consumption.csv'
# consumption_df = preparer.load_raw_consumption(raw_path=test_file)
# UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format. df = pd.read_csv(
# L'avertissement apparaît parce que pandas ne peut pas deviner automatiquement le format des dates et utilise dateutil (plus lent) au lieu d'un parseur optimisé.
# Pour éviter cela, vous pouvez spécifier le format de date attendu dans votre CSV en utilisant le paramètre `parse_dates` et `date_parser` de `pd.read_csv()`. Par exemple:
# date_parser = lambda x: pd.to_datetime(x, format='%Y-%m-%d %H:%M:%S')
# df = pd.read_csv('file.csv', parse_dates=['date_column'], date_parser=date_parser)
# Cependant, nous souhaitons que le code soit flexible et puisse gérer différents formats de date, donc nous avons laissé pandas inférer le format. L'avertissement est informatif mais n'empêche pas le chargement des données.

try:
    # Si le fichier existe, essayer de le charger
    raw_file = config.get('data', {}).get('raw_path')
    print(f'Chargement de: {raw_file}')
    consumption_df = preparer.load_raw_consumption()  # Pas de paramètre = utilise config
    print(f'✅ Données consommation chargées: {len(consumption_df)} enregistrements')
    print(f'Colonnes: {list(consumption_df.columns)}')
    display(consumption_df.head())
except FileNotFoundError as e:
    print(f'⚠️ Fichier non trouvé: {e}')
    print('→ Utilisez un fichier existant ou modifiez la config')
except Exception as e:
    print(f'❌ Erreur: {e}')

C:\_sources\Jenedai\jinsudai\src\ml\consumption\consumption_preparer.py:78: UserWarning: Parsing dates in %Y-%m-%d %H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df = pd.read_csv(
2026-06-06 18:26:11,356 - INFO - Données consommation chargées: 1488 enregistrements


Chargement de: ../data/test/raw_consumption.csv
✅ Données consommation chargées: 1488 enregistrements
Colonnes: ['Horodate', 'Valeur']


,Horodate,Valeur
0,2024-01-01 00:00:00,524.835708
1,2024-01-01 00:30:00,493.086785
2,2024-01-01 01:00:00,482.384427
3,2024-01-01 01:30:00,526.151493
4,2024-01-01 02:00:00,388.292331


## 3. Test avec données complètes (si disponibles)

In [12]:
# Chemins des fichiers depuis la configuration
WEATHER_FILE = config.get('data', {}).get('weather_file', '../data/processed/weather.parquet')
HOLIDAYS_FILE = config.get('data', {}).get('holidays_file', '../data/processed/holidays.parquet')
TRAIN_FILE = config.get('data', {}).get('train_path', '../data/processed/consumption/train.parquet')

# Vérifier que les fichiers existent
files_exist = {
    'weather': Path(WEATHER_FILE).exists(),
    'holidays': Path(HOLIDAYS_FILE).exists()
}
print(f'Fichiers disponibles: {files_exist}')

Fichiers disponibles: {'weather': True, 'holidays': True}


In [13]:
# Si les fichiers existent, tester le pipeline complet
if all(files_exist.values()):
    print('\n>>> Test du pipeline complet (tout depuis config)...')
    try:
        # Tous les chemins viennent de la config
        features_df = preparer.prepare()  # Aucun paramètre = tout depuis config
        print(f'✅ Features générées: {len(features_df)} enregistrements')
        print(f'Colonnes: {list(features_df.columns)}')
        display(features_df.head())
        print(f'Fichier sauvegardé: {TRAIN_FILE}')
    except Exception as e:
        print(f'❌ Erreur dans le pipeline: {e}')
else:
    print('⚠️ Tous les fichiers ne sont pas disponibles pour le test complet')


>>> Test du pipeline complet (tout depuis config)...


C:\_sources\Jenedai\jinsudai\src\ml\consumption\consumption_preparer.py:78: UserWarning: Parsing dates in %Y-%m-%d %H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df = pd.read_csv(
2026-06-06 18:26:11,511 - INFO - Données consommation chargées: 1488 enregistrements
2026-06-06 18:26:11,531 - INFO - Données météo chargées: 1488 enregistrements
2026-06-06 18:26:11,550 - INFO - Données calendrier chargées: 1488 enregistrements
2026-06-06 18:26:11,587 - INFO - Datasets fusionnés: 1488 enregistrements
2026-06-06 18:26:11,591 - INFO - ✅ Validation contre configuration réussie


Loading weather data from: ../data/processed/weather.parquet


2026-06-06 18:26:11,627 - INFO - ✅ Features consommation sauvegardées: ..\data\dev\train_consumption.parquet


✅ Features générées: 1488 enregistrements
Colonnes: ['Horodate', 'Valeur', 'temperature_2m_mean', 'relative_humidity_mean', 'precipitation_sum', 'is_vacances', 'nom_vacances', 'jour de la semaine', 'jour férié']


,Horodate,Valeur,temperature_2m_mean,relative_humidity_mean,precipitation_sum,is_vacances,nom_vacances,jour de la semaine,jour férié
0,2024-01-01 00:00:00,524.835708,2.371718,82.637067,0.023052,1,Noël,Monday,1
1,2024-01-01 00:30:00,493.086785,16.312921,61.536840,0.043461,1,Noël,Monday,1
2,2024-01-01 01:00:00,482.384427,7.240709,75.434794,0.013480,1,Noël,Monday,1
3,2024-01-01 01:30:00,526.151493,22.790996,71.998104,0.013910,1,Noël,Monday,1
4,2024-01-01 02:00:00,388.292331,7.178762,72.640201,0.112762,1,Noël,Monday,1


Fichier sauvegardé: ../data/dev/train_consumption.parquet


## 5. Résumé

In [14]:
print('\n=== RÉSUMÉ ===')
print('✅ Configuration chargée avec succès')
print('✅ ConsumptionDataPreparer initialisé')
print('✅ Support des environnements (dev/test/prod) fonctionnel')
print('\nProchaines étapes:')
print('- Exécuter avec ENV=dev pour le développement')



=== RÉSUMÉ ===
✅ Configuration chargée avec succès
✅ ConsumptionDataPreparer initialisé
✅ Support des environnements (dev/test/prod) fonctionnel

Prochaines étapes:
- Exécuter avec ENV=dev pour le développement
